# 从 Attention 到 TTT：线性化之路（4/5）—— 两次平均：从 Attention 走到 TTT 读出

第 3 册结束时，V 被压进了状态 $W$，但 K 还在，输出式

$$o = \sum_{i=1}^{n} a_i \mu_W(k_i)$$

仍然是 Softmax Attention 的形式，复杂度并没有降低。这一册用**两次平均近似**把它化简。

## 1. 第一次平均场近似：把 μ 挪进求和号

第一个近似假设 $\mu_W$ 可以拿到加权平均里面去：

$$\sum_{i=1}^{n} a_i \mu_W(k_i) \;\approx\; \mu_W\Big(\sum_{i=1}^{n} a_i k_i\Big)$$

这样核心计算变成了 $\sum a_i k_i$ —— 一个 **K=V** 的 Softmax Attention。

> 注意：当 $\mu_W$ 是**线性模型**时，这个等式是**精确恒等**，不再只是近似（第 5 册展开）。

## 2. 第二次加权平均近似：历史键的平均 ≈ 查询本身

权重满足 $a_i \propto e^{q \cdot k_i}$。如果假设 $k_1,\dots,k_n$ 比较**各向同性**，
更准确地说：它们垂直于 $q$ 的分量加权平均近似为零，那么

$$\sum_{i=1}^{n} a_i k_i \;\approx\; \lambda q$$

其中 $\lambda$ 是个标量。于是 $\mu_W(\cdot)$ 的输入可以换成 $\lambda q$，
把 $\lambda$ 吸收进参数里，得到非常干净的读出：

$$o = \mu_W(q)$$

连同第 3 册的递推式 (6)，就组成了一个以 $(q, (k_1,v_1),\dots,(k_n,v_n))$ 为输入、
以 $W$ 为状态的 RNN：**完整的 K、V 不再出现，全部被逐步压缩进参数 $W$**。

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
n, d = 8, 4
q = rng.standard_normal(d)
q = q / np.linalg.norm(q)

def weighted_keys(K, q):
    score = K @ q
    score -= score.max()
    a = np.exp(score); a /= a.sum()
    return a, a @ K

# 情形 A: 键各向同性(随机方向) —— 第二次近似应当成立
K_iso = rng.standard_normal((n, d))
a, mean_k = weighted_keys(K_iso, q)
lam = mean_k @ q                       # 把 mean_k 投影到 q 方向
resid_iso = np.linalg.norm(mean_k - lam * q)

# 情形 B: 键都贴着某个非 q 方向 —— 近似会明显变坏
K_bias = rng.standard_normal((n, d))
bias = rng.standard_normal(d); bias -= (bias @ q) * q   # 垂直于 q 的偏置
K_bias = K_bias + 1.5 * bias
a2, mean_k2 = weighted_keys(K_bias, q)
lam2 = mean_k2 @ q
resid_bias = np.linalg.norm(mean_k2 - lam2 * q)

print(f"各向同性组: 残差 ‖Σa k - λq‖ = {resid_iso:.3f}")
print(f"带偏置组  : 残差 ‖Σa k - λq‖ = {resid_bias:.3f}")
print()
print("解释: 残差越小, Σa k ≈ λq 这个近似越靠谱; 键一旦有垂直于 q 的结构性偏置, 近似就松了")

## 3. 三步近似复盘：每一步放弃了什么，收获了什么

| 步骤 | 操作 | 放弃了什么 | 收获了什么 |
|---|---|---|---|
| 第一步 | 用 $\mu_W(k_i)$ 预测替代硬背的 $v_i$ | 逐点精确配对 | 消去 V Cache |
| 第二步 | 平均场 $\sum a_i \mu(k_i) pprox \mu(\sum a_i k_i)$ | 数学上的近似程度 | Attention 从输出侧挪到输入侧 |
| 第三步 | $\sum a_i k_i pprox \lambda q$，输入换成 $q$ | 历史键的结构信息 | 彻底变成线性复杂度 RNN |

文章特别指出：**最成问题的是第三步**。K=V 的 Attention 并不总近似于 $\lambda q$——
反例就是 MLA、DSV4 这类 K=V 的 MQA，如果只相当于 $q$，根本建模不了 token 之间的交互。

## 4. 为什么实践中要 Hybrid（混合）

第三步的假设在真实数据上很难完全吻合，误差会累积。主流做法是**定时补回 Softmax Attention**
来弥补这一差距：一部分层走线性 RNN，一部分层保留全注意力。这正是当前 Hybrid 架构的动机。

## 5. 小结（4/5）

- 平均场近似把 $\mu$ 挪进求和号，得到 K=V 的注意力；
- 各向同性假设下 $\sum a_i k_i pprox \lambda q$，读出化简为 $o = \mu_W(q)$；
- 三步近似里第三步最激进，也是信息损失的主要来源；
- 工程上用 Hybrid 混合层补回全注意力，修复近似缺口。

→ **下一册**：把 $\mu_W$ 简化成线性函数 $Wk$，整条链落到 DeltaNet；
再换解析解累加器，就得到 MesaNet，并看看两者状态大小的差别。